# 07 · Model Building

## Objective
Train and compare classification models **with identical preprocessing and identical CV folds**,
starting from a baseline, then tune the most promising models.

Rules followed:
- Only the **training set** is used (5-fold stratified cross-validation). The test set is reserved for notebook 08.
- Each model is a full `Pipeline` (preprocessing + model), so the preprocessor is re-fitted inside every
  CV fold → no leakage.
- Original features only (decision from notebook 06).

In [1]:
import sys
from pathlib import Path

# Make the project root importable so we can use src/churn_utils.py
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import time

import joblib
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, precision_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate

from src.churn_utils import MODELS_DIR, RANDOM_STATE, build_pipeline, load_split

X_train, X_test, y_train, y_test = load_split()   # X_test / y_test are NOT used in this notebook
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
# zero_division=0: the Dummy model never predicts churn, so its precision is 0 (not undefined)
SCORING = {"accuracy": "accuracy", "precision": make_scorer(precision_score, zero_division=0),
           "recall": "recall", "f1": "f1", "roc_auc": "roc_auc", "average_precision": "average_precision"}
print("Train:", X_train.shape, "| churn rate:", round(y_train.mean(), 4))

Train: (5634, 19) | churn rate: 0.2654


## 1. Model candidates

| Model | Why it is included |
|---|---|
| **Dummy (most frequent)** | Baseline: always predicts "No churn". Any useful model must beat it. |
| **Logistic Regression** | Simple, fast, interpretable (coefficients ≈ effect direction). Strong baseline for tabular data. |
| **Random Forest** | Captures non-linear effects and interactions (e.g. contract × tenure) without manual features. |
| **Gradient Boosting** | Usually strong on tabular data; builds trees sequentially to correct earlier errors. |

All are standard scikit-learn models — no extra libraries needed. Parallelism (`n_jobs=-1`) is set only on the
CV / grid-search loop, not inside the models, to avoid nested parallel jobs.

**Primary selection metric: ROC-AUC** (does the model rank churners above non-churners?). It is
threshold-independent, which suits us because the decision threshold is chosen separately in notebook 08.
PR-AUC, recall, precision and F1 are reported alongside.

**Selection rule (decided before looking at results):** choose the model with the best mean CV ROC-AUC;
if the gap to a simpler model is smaller than the fold-to-fold standard deviation, prefer the simpler,
more interpretable model.

## 2. Baseline and default models (5-fold CV)

In [2]:
def cv_summary(pipeline, name):
    """Mean of each CV metric for one pipeline."""
    scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=SCORING, n_jobs=-1)
    summary = {metric: scores[f"test_{metric}"].mean() for metric in SCORING}
    summary["roc_auc_std"] = scores["test_roc_auc"].std()
    return pd.Series(summary, name=name)

default_models = {
    "Dummy (most frequent)": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
}

default_results = pd.DataFrame([cv_summary(build_pipeline(model), name)
                                for name, model in default_models.items()])
default_results.round(4)

,accuracy,precision,recall,f1,roc_auc,average_precision,roc_auc_std
Dummy (most frequent),0.7346,0.0000,0.0000,0.0000,0.5000,0.2654,0.0000
Logistic Regression,0.8023,0.6530,0.5445,0.5932,0.8462,0.6615,0.0126
Random Forest,0.7879,0.6315,0.4836,0.5474,0.8228,0.6159,0.0119
Gradient Boosting,0.8033,0.6622,0.5291,0.5880,0.8483,0.6677,0.0126


**Observations**
- The **Dummy** model reaches **73.5% accuracy** while catching **zero** churners (recall 0, ROC-AUC 0.5).
  This is the clearest proof that accuracy alone is misleading here.
- Default Logistic Regression (ROC-AUC 0.846) and Gradient Boosting (0.848) perform similarly.
- Default Random Forest is weaker (0.823): fully grown trees overfit the training folds — tuning should help.
- At the default 0.5 threshold, all models catch only about **half of the churners** (recall 0.48–0.54).
  The threshold will need adjusting for the business goal (notebook 08).

## 3. Hyperparameter tuning

Tuning is justified because default settings are rarely optimal (e.g. an unrestricted Random Forest
can overfit). Grids are kept **small and meaningful** to limit compute and the risk of overfitting
the CV score.

| Model | Tuned parameters | What they control |
|---|---|---|
| Logistic Regression | `C`, `class_weight` | Regularisation strength; whether churn errors are up-weighted |
| Random Forest | `max_depth`, `min_samples_leaf`, `max_features`, `class_weight` | Tree complexity / overfitting; imbalance |
| Gradient Boosting | `n_estimators`, `learning_rate`, `max_depth` | Number and size of boosting steps |

`GridSearchCV` uses the **same 5 folds** and `scoring="roc_auc"`. With `refit=True` the best
configuration is re-trained on the full training set.

In [3]:
param_grids = {
    "Logistic Regression": (
        LogisticRegression(max_iter=1000),
        {"model__C": [0.01, 0.1, 1, 10], "model__class_weight": [None, "balanced"]},
    ),
    "Random Forest": (
        RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE),
        {"model__max_depth": [6, 10, None], "model__min_samples_leaf": [1, 5, 10],
         "model__max_features": ["sqrt", 0.5], "model__class_weight": [None, "balanced"]},
    ),
    "Gradient Boosting": (
        GradientBoostingClassifier(random_state=RANDOM_STATE),
        {"model__n_estimators": [100, 200, 300], "model__learning_rate": [0.03, 0.1],
         "model__max_depth": [2, 3]},
    ),
}

searches = {}
for name, (model, grid) in param_grids.items():
    start = time.time()
    search = GridSearchCV(build_pipeline(model), grid, cv=cv, scoring="roc_auc", n_jobs=-1)
    search.fit(X_train, y_train)
    searches[name] = search
    best_params = {k.replace("model__", ""): v for k, v in search.best_params_.items()}
    print(f"{name:<20} best CV ROC-AUC = {search.best_score_:.4f}  "
          f"({time.time() - start:.0f}s)  {best_params}")

Logistic Regression  best CV ROC-AUC = 0.8464  (1s)  {'C': 10, 'class_weight': None}


Random Forest        best CV ROC-AUC = 0.8479  (24s)  {'class_weight': None, 'max_depth': 6, 'max_features': 0.5, 'min_samples_leaf': 10}


Gradient Boosting    best CV ROC-AUC = 0.8500  (8s)  {'learning_rate': 0.03, 'max_depth': 2, 'n_estimators': 300}


### Tuned models: full CV metrics
Re-run CV on each tuned configuration to get every metric on the same folds
(metrics at the default 0.5 threshold; threshold choice comes in notebook 08).

In [4]:
tuned_results = pd.DataFrame([cv_summary(search.best_estimator_, f"{name} (tuned)")
                              for name, search in searches.items()])
all_results = pd.concat([default_results, tuned_results])
all_results.sort_values("roc_auc", ascending=False).round(4)

,accuracy,precision,recall,f1,roc_auc,average_precision,roc_auc_std
Gradient Boosting (tuned),0.8056,0.6725,0.5224,0.5877,0.8500,0.6698,0.0124
Gradient Boosting,0.8033,0.6622,0.5291,0.5880,0.8483,0.6677,0.0126
Random Forest (tuned),0.8030,0.6706,0.5077,0.5777,0.8479,0.6645,0.0102
Logistic Regression (tuned),0.8039,0.6559,0.5492,0.5973,0.8464,0.6607,0.0128
Logistic Regression,0.8023,0.6530,0.5445,0.5932,0.8462,0.6615,0.0126
Random Forest,0.7879,0.6315,0.4836,0.5474,0.8228,0.6159,0.0119
Dummy (most frequent),0.7346,0.0000,0.0000,0.0000,0.5000,0.2654,0.0000


**Observations**
- **Gradient Boosting (tuned)**: best mean ROC-AUC **0.8500** (300 shallow trees, depth 2, learning rate 0.03).
- **Random Forest (tuned)**: improved the most (0.823 → **0.848**) by limiting depth to 6 and requiring ≥10 samples per leaf.
  This confirms the default forest was overfitting.
- **Logistic Regression (tuned)**: **0.8464**, practically unchanged from default (C = 10, i.e. weak regularisation).
- `class_weight="balanced"` was **not** selected for any model. This is expected: ROC-AUC measures ranking, and
  class weights mainly shift predicted probabilities (similar to moving the threshold). The imbalance is handled
  through threshold choice in notebook 08 instead.
- All tuned models lie within **0.0036 ROC-AUC** of each other, much less than the fold-to-fold std (~0.012).
  The models are effectively tied, which suggests the main limit is the information in the features,
  not the algorithm.

## 4. Save the tuned candidates

The tuned pipelines (fitted on the full training set by `GridSearchCV`) are saved so notebook 08
can compare them in detail and evaluate the chosen one on the test set.

In [5]:
MODELS_DIR.mkdir(exist_ok=True)
tuned_models = {name: search.best_estimator_ for name, search in searches.items()}
joblib.dump(tuned_models, MODELS_DIR / "tuned_candidates.pkl")
print("Saved:", list(tuned_models))

Saved: ['Logistic Regression', 'Random Forest', 'Gradient Boosting']


## Conclusion

| Model (tuned) | CV ROC-AUC | CV PR-AUC | CV recall @0.5 |
|---|---|---|---|
| Gradient Boosting | 0.8500 ± 0.012 | 0.670 | 0.522 |
| Random Forest | 0.8479 ± 0.010 | 0.665 | 0.508 |
| Logistic Regression | 0.8464 ± 0.013 | 0.661 | 0.549 |
| Dummy baseline | 0.5000 | 0.265 | 0.000 |

- All models clearly beat the baseline.
- Applying the **pre-defined selection rule**: Gradient Boosting's lead over Logistic Regression (0.0036)
  is smaller than one standard deviation, so the **simpler and more interpretable Logistic Regression is preferred**.
  Notebook 08 checks this choice with a detailed comparison (confusion matrices, ROC/PR curves) on
  cross-validated predictions before the single, final test-set evaluation.
- Tuned candidates saved to `models/tuned_candidates.pkl`.